####  a)Extraction de l'annee de sortie des Films

In [27]:
#Import des Librairies Necessaires
from pyspark.sql import SparkSession, DataFrame 
from pyspark.sql.functions import col, from_json, explode, regexp_replace
from pyspark.sql.types import ArrayType, StructType, StructField, IntegerType, StringType

# Creation Session Spark
spark = SparkSession.builder \
    .appName("PySpark_movies_correction") \
    .master("local[*]") \
    .getOrCreate()


df= spark.read.option("header", "true").option("inferSchema", "true").csv("movies.csv") #Lecture des donnees dans le fichier movies.csv et tranformation en DF spark en considerant la premiere ligne comme noms des colonnes
df.createOrReplaceTempView("movies")  # Creation d'une table temporaire pour permettre interrogation du DataFrame par SQL

year = spark.sql("""
    SELECT
        id,
        original_title,
        regexp_extract(release_date, '^(\\\\d{4})', 1) AS year    /*on prend les 4 premier chiffre dans la colonne year */
    FROM movies
""")

year.show()

+-----+--------------------+----+
|   id|      original_title|year|
+-----+--------------------+----+
|  862|           Toy Story|1995|
| 8844|             Jumanji|1995|
|15602|    Grumpier Old Men|1995|
|31357|   Waiting to Exhale|    |
|11862|Father of the Bri...|1995|
|  949|                Heat|1995|
|11860|             Sabrina|1995|
|45325|        Tom and Huck|1995|
| 9091|        Sudden Death|1995|
|  710|           GoldenEye|1995|
| 9087|The American Pres...|1995|
|12110|Dracula: Dead and...|1995|
|21032|               Balto|1995|
|10858|               Nixon|1995|
| 1408|    Cutthroat Island|1995|
|  524|              Casino|1995|
| 4584|Sense and Sensibi...|1995|
|    5|          Four Rooms|1995|
| 9273|Ace Ventura: When...|1995|
|11517|         Money Train|1995|
+-----+--------------------+----+
only showing top 20 rows



#### b) Comptage des 10 genres de films les plus populaires.

In [28]:
# Définir le schéma JSON
genre_schema = ArrayType(
    StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])
)

# Créer un nouveau DataFrame avec array struct
df_parsed = df.withColumn(
    "genres_parsed",
    from_json(
        regexp_replace(col("genres"), "'", '"'),  # remplacer ' par "
        genre_schema
    )
)

#Les genres les plus frequent
most_pop_movies = (
    df_parsed
        .select(explode(col("genres_parsed")).alias("g"))
        .groupBy(col("g.name").alias("genre"))
        .count()
        .orderBy(col("count").desc())
)

most_pop_movies.show(10)


+---------------+-----+
|          genre|count|
+---------------+-----+
|          Drama|20239|
|         Comedy|13123|
|       Thriller| 7612|
|        Romance| 6722|
|         Action| 6581|
|         Horror| 4656|
|          Crime| 4294|
|    Documentary| 3929|
|      Adventure| 3477|
|Science Fiction| 3041|
+---------------+-----+
only showing top 10 rows



#### c)Les films appartenant a aumoins 3 genres

In [29]:
films_multi_genres = spark.sql("""
    SELECT
        id,
        original_title,
        (
            length(genres) - length(regexp_replace(genres, 'name', ''))
        ) / length('name') AS nb_genres
    FROM movies
    WHERE (
        length(genres) - length(regexp_replace(genres, 'name', ''))
    ) / length('name') >= 3
""")

films_multi_genres.show()

+------+--------------------+---------+
|    id|      original_title|nb_genres|
+------+--------------------+---------+
|   862|           Toy Story|      3.0|
|  8844|             Jumanji|      3.0|
| 31357|   Waiting to Exhale|      3.0|
|   949|                Heat|      4.0|
| 45325|        Tom and Huck|      4.0|
|  9091|        Sudden Death|      3.0|
|   710|           GoldenEye|      3.0|
|  9087|The American Pres...|      3.0|
| 21032|               Balto|      3.0|
|  9273|Ace Ventura: When...|      3.0|
| 11517|         Money Train|      3.0|
|  8012|          Get Shorty|      3.0|
|  9691|           Assassins|      4.0|
| 12665|              Powder|      4.0|
|  9263|        Now and Then|      3.0|
|   902|La Cité des Enfan...|      3.0|
|    63|      Twelve Monkeys|      3.0|
|  9598|                Babe|      4.0|
| 47018|          Carrington|      3.0|
|139405|Across the Sea of...|      4.0|
+------+--------------------+---------+
only showing top 20 rows



#### d)Les 5 annees les plus productives

In [30]:
year.createOrReplaceTempView("movies_with_year")

CTE = spark.sql("""
    WITH films_par_annee AS (
        SELECT
            year,
            COUNT(*) AS nb_films
        FROM movies_with_year
        WHERE year != '' /* on n'inclut pas le nombre de filme sans annee  */
        GROUP BY year
    )
    SELECT *
    FROM films_par_annee
    ORDER BY nb_films DESC
    LIMIT 5
""")

CTE.show()

+----+--------+
|year|nb_films|
+----+--------+
|2014|    1865|
|2015|    1802|
|2013|    1763|
|2012|    1599|
|2011|    1543|
+----+--------+



#### e)Genres dominants par décennie.

In [31]:
movies_with_genres = spark.sql("""
    SELECT
        id,
        genres
    FROM movies
""")


classement = spark.sql("""
    WITH joined AS (
        SELECT
            m.id,
            w.year,
            from_json(m.genres, 'array<struct<id:int,name:string>>') AS genres_parsed      /*Convertir la chaîne JSON en array de structs */
        FROM movies m
        JOIN movies_with_year w
        ON m.id = w.id
        WHERE w.year != ''
    ),
    exploded AS (
        SELECT
            CAST(year AS INT) - (CAST(year AS INT) % 10) AS decade,                           /*Explosion de Genre*/
            explode(transform(genres_parsed, x -> x.name)) AS genre
        FROM joined
    ),
    genre_count AS (
        SELECT
            decade,
            genre,                                                  /* Pour chaque Decennie, on compte le  genre*/
            COUNT(*) AS nb_films
        FROM exploded
        GROUP BY decade, genre
    ),
    ranked AS (
        SELECT
            decade,
            genre,
            nb_films,                                              /* Avec le compte fait, on fait un classement du nombre de genre pour chauqe decenie*/
            ROW_NUMBER() OVER (PARTITION BY decade ORDER BY nb_films DESC) AS rank
        FROM genre_count
    )
    SELECT
        decade,
        genre,
        nb_films                                                     /* on retient le premier genre pour chaque decennie*/
    FROM ranked
    WHERE rank = 1
    ORDER BY decade
""")


classement.show()


+------+-----------+--------+
|decade|      genre|nb_films|
+------+-----------+--------+
|  1870|Documentary|       2|
|  1880|Documentary|       4|
|  1890|Documentary|      25|
|  1900|    Fantasy|      29|
|  1910|      Drama|      75|
|  1920|      Drama|     246|
|  1930|      Drama|     656|
|  1940|      Drama|     721|
|  1950|      Drama|     973|
|  1960|      Drama|    1109|
|  1970|      Drama|    1368|
|  1980|      Drama|    1420|
|  1990|      Drama|    2353|
|  2000|      Drama|    4877|
|  2010|      Drama|    4899|
|  2020|     Action|       1|
+------+-----------+--------+



#### f)Détection des titres dupliqués.

In [32]:
#Identification des Titres presents au moins en double + le nombre de leur presence + id

Detection_doublons = spark.sql("""
    SELECT
        original_title,
        COUNT(*) AS nb_versions,
        collect_list(id) AS ids_list
    FROM movies
    GROUP BY original_title
    HAVING COUNT(*) >= 2
    ORDER BY nb_versions DESC
""")
          
Detection_doublons.show(20, truncate=50)

+----------------------------------------+-----------+--------------------------------------------------+
|                          original_title|nb_versions|                                          ids_list|
+----------------------------------------+-----------+--------------------------------------------------+
|                                    NULL|         32|                                 [ Chochana naïve]|
|                                       0|         16|[[], [{'name': 'Knightsbridge Films', 'id': 205...|
|                                      []|          9|[2004-06-23, 2011-10-20, 0.081204, 1979-06-06, ...|
|                     Alice in Wonderland|          8|[12092, 30923, 12155, 25694, 34573, 35109, 8730...|
|                                  Hamlet|          8|[10549, 23383, 10688, 10264, 125705, 28238, 106...|
|                       A Christmas Carol|          7| [25842, 13189, 17979, 16716, 45697, 47257, 28769]|
|                              Cinderella|    